# RNNs and LSTMs, Explained Simply
Networks with memory: forecasting daily park visitors with a recurrent network and an LSTM, in PyTorch. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
Three years of daily visitor counts at a city park (made up for this video). Busy weekends, busy summers, and slow growth.

### Three years of park visitors
- `pd.read_csv(url)`: load the table straight from the web
- `df.head(7)`: the first week: Saturday and Sunday are much busier

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "rnns-and-lstms/park_visitors.csv")
df = pd.read_csv(url)
print(len(df), "days")
print(df.head(7))

### Plot the visitors
- `left`: every day for three years: summer peaks
- `right`: the first four weeks: a weekly rhythm

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
a1.plot(df["visitors"]); a1.set_title("three years")
a2.plot(df["visitors"][:28], marker="o"); a2.set_title("first four weeks")
for a in (a1, a2):
    a.set_xlabel("day"); a.set_ylabel("visitors")
plt.show()

## 2. Simple guesses first
Before any network, two simple forecasts for tomorrow: **the same as yesterday**, and **the same as this day last week**. Error = mean absolute error (MAE): the average size of the miss, in visitors.

### Two simple forecasts
- `cut`: the last 20% of days are the test period
- `v[cut - 1:-1]`: for each test day, the day before
- `v[cut - 7:-7]`: for each test day, the same weekday a week earlier

In [ ]:
import numpy as np

v = df["visitors"].values
cut = int(len(v) * 0.8)
actual = v[cut:]
yesterday = v[cut - 1:-1]
last_week = v[cut - 7:-7]
print("same as yesterday:", round(np.abs(actual - yesterday).mean(), 1))
print("same as last week:", round(np.abs(actual - last_week).mean(), 1))

## 3. A recurrent neuron by hand
A recurrent network reads one day at a time and keeps a **hidden state** h, its memory. **h_new = tanh(w_x x input + w_h x h_old + b)**. With w_x = 0.5, w_h = 0.8, b = 0 and inputs 1, 0, 0, the memory fades: 0.462, 0.354, 0.276.

### Three steps by hand
- `h = 0.0`: the memory starts empty
- `w_h * h`: yesterday's memory feeds into today
- `math.tanh`: squash the result to between -1 and 1

In [ ]:
import math

w_x, w_h, b = 0.5, 0.8, 0.0
h = 0.0
for day, x in enumerate([1, 0, 0], start=1):
    h = math.tanh(w_x * x + w_h * h + b)
    print(f"day {day}: input {x}, memory h = {h:.3f}")

## 4. Windows of 14 days
Turn the series into examples: the input is 14 days in a row, the answer is the next day. Divide by 1,000 so the numbers are small. Train on the first 80% of time, test on the last 20%, never shuffling time.

### Make the windows
- `s[i:i + W]`: 14 days in a row
- `y = s[W:]`: the day after each window
- `.unsqueeze(-1)`: one number per day (windows, days, features)
- `split = cut - W`: so the test answers are exactly the same days as the simple forecasts

In [ ]:
import torch

W = 14
s = v / 1000
X = np.stack([s[i:i + W] for i in range(len(s) - W)])
y = s[W:]
X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(y, dtype=torch.float32)
split = cut - W
X_tr, y_tr, X_te, y_te = X[:split], y[:split], X[split:], y[split:]
print(X_tr.shape, X_te.shape)

## 5. The model and training
An RNN (or LSTM) reads the 14 days one at a time; after the last day, an ordinary layer turns its memory into a forecast.

### The forecaster
- `cell`: nn.RNN or nn.LSTM, passed in
- `hidden_size=32`: a memory of 32 numbers
- `batch_first=True`: the data is (windows, days, features)
- `states[:, -1]`: the memory after the last day
- `nn.Linear(32, 1)`: memory -> one forecast

In [ ]:
import torch.nn as nn

class Forecaster(nn.Module):
    def __init__(self, cell):
        super().__init__()
        self.rnn = cell(input_size=1, hidden_size=32, batch_first=True)
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        states, _ = self.rnn(x)
        return self.out(states[:, -1]).squeeze(-1)

### A training function
- `nn.MSELoss()`: mean squared error: for numbers, not classes
- `the loop`: mini-batches of 32 windows, as usual
- `* 1000`: back to visitors
- `returns`: the test MAE in visitors, and the forecasts

In [ ]:
def fit(model, epochs=60):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_fn = nn.MSELoss()
    for epoch in range(epochs):
        order = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 32):
            idx = order[i:i + 32]
            optimizer.zero_grad()
            loss_fn(model(X_tr[idx]), y_tr[idx]).backward()
            optimizer.step()
    with torch.no_grad():
        pred = model(X_te)
    return (pred - y_te).abs().mean().item() * 1000, pred * 1000

### Train an RNN
- `nn.RNN`: the simple recurrent network from the by-hand example, with 32 memory numbers

In [ ]:
torch.manual_seed(0)
rnn = Forecaster(nn.RNN)
mae_rnn, _ = fit(rnn)
print(f"RNN error: {mae_rnn:.1f} visitors")

## 6. The vanishing gradient
To learn from a day long ago, the slope has to travel back through every step, and it's multiplied at each one. Multiply by less than 1 many times and it vanishes: 0.8^20 = 0.012, 0.8^50 = 0.000014.

### Measure it
- `x.grad`: how much the final memory depends on each day's input
- `plt.semilogy`: a log scale: each grid line is 10 times smaller
- `the picture`: the slope from day 1 is billions of times smaller than from day 50

In [ ]:
torch.manual_seed(0)
cell = nn.RNN(1, 32, batch_first=True)
x = torch.randn(1, 50, 1, requires_grad=True)
states, _ = cell(x)
states[0, -1].sum().backward()
slope = x.grad[0, :, 0].abs()
print("slope from day 50:", f"{slope[-1]:.2g}")
print("slope from day 1: ", f"{slope[0]:.2g}")
plt.figure(figsize=(9, 3.5))
plt.semilogy(range(1, 51), slope)
plt.xlabel("day"); plt.ylabel("slope (log scale)"); plt.show()

## 7. The LSTM
An LSTM adds a **cell state**, a conveyor belt of memory, and three gates: **forget**, **input** and **output**. **c_new = forget x c_old + input x candidate**. The network learns gate values; a forget gate near 1 keeps a memory for a long time.

### Train an LSTM
- `nn.LSTM`: swap one word: same forecaster, LSTM memory
- `the output`: all four errors, in visitors

In [ ]:
torch.manual_seed(0)
lstm = Forecaster(nn.LSTM)
mae_lstm, pred = fit(lstm)
print(f"same as yesterday: {np.abs(actual - yesterday).mean():.1f}")
print(f"same as last week: {np.abs(actual - last_week).mean():.1f}")
print(f"RNN:               {mae_rnn:.1f}")
print(f"LSTM:              {mae_lstm:.1f}")

### Forecasts vs reality
- `[-56:]`: the last 56 days: eight weeks
- `the picture`: the forecast follows the weekend peaks closely

In [ ]:
plt.figure(figsize=(12, 3.5))
plt.plot(actual[-56:], marker="o", label="actual")
plt.plot(pred[-56:].numpy(), marker="x", label="LSTM forecast")
plt.xlabel("day (last 8 weeks)"); plt.ylabel("visitors"); plt.legend()
plt.show()

## 8. Mini project: forecast tomorrow
Use the last 14 days of data to forecast the day after the data ends.

### Forecast tomorrow
- `s[-W:]`: the final 14 days
- `.view(1, W, 1)`: one window, 14 days, one number per day

In [ ]:
last14 = torch.tensor(s[-W:], dtype=torch.float32).view(1, W, 1)
with torch.no_grad():
    tomorrow = lstm(last14).item() * 1000
print("last 7 days:", v[-7:])
print(f"forecast for tomorrow: {tomorrow:.0f} visitors")

## Try it yourself
1. Work out the RNN by hand for inputs 1, 1, 1. Then check it with the code.
2. Change the window from 14 days to 28. Does the LSTM get better?
3. Change the hidden size from 32 to 8. What happens to the error?